# Otimização de Infraestrutura em Estação de Reabastecimento de Hidrogênio (HRS)
## Otimização de Rotas de Tubulações e Redes de Automação SCADA via Algoritmos de Kruskal e Prim

---

### 1. Contexto Industrial e Engenharia de Processos
Em uma Estação de Reabastecimento de Hidrogênio (HRS), o investimento na **infraestrutura de tubulação (alta e baixa pressão)** e no **cabeamento de sinal/comunicação SCADA** representa uma fração crítica do custo de capital (*CAPEX*) e da segurança operacional.

O objetivo fundamental da modelagem via **Matemática Discreta / Teoria dos Grafos** neste cenário é encontrar a **Árvore Geradora Mínima (Minimum Spanning Tree - MST)**. A MST garante que todos os equipamentos da estação (eletrolisador, compressores, armazenamento e dispensers) estejam plenamente interconectados à central de controle, minimizando o comprimento total de tubos/cabos sem criar laços fechados redundantes.

#### Modelagem Matemática Formal
Dado um grafo não direcionado e ponderado $G = (V, E, w)$:
- $V$: Conjunto de vértices representando os **equipamentos operacionais e instrumentos** ($|V| = n$).
- $E$: Conjunto de arestas representando os **leitos de cabos e tubulações viáveis** ($|E| = m$).
- $w: E \to \mathbb{R}^+$: Função de peso que atribui a distância física em metros $w(e)$ a cada aresta $e \in E$.

Uma Árvore Geradora $T = (V, E_T)$ é um subgrafo conexo acíclico contendo todos os vértices de $V$, com $|E_T| = |V| - 1$. A MST busca minimizar a função objetivo:

$$W(T) = \sum_{e \in E_T} w(e)$$

---
### 2. Fundamentação dos Algoritmos

#### A. Algoritmo de Kruskal (Abordagem Centrada em Arestas)
- **Estratégia Gulosa (*Greedy*):** Ordena todas as arestas do grafo em ordem crescente de peso. Adiciona iterativamente a aresta de menor peso à solução, desde que ela não forme um ciclo com as arestas já selecionadas.
- **Estrutura de Dados:** Utiliza **Conjuntos Disjuntos (*Disjoint Set Union / Union-Find*)** com otimizações de *Compressão de Caminho* e *União por Rank*.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |E|)$ ou $\mathcal{O}(|E| \log |V|)$.

#### B. Algoritmo de Prim (Abordagem Centrada em Vértices)
- **Estratégia Gulosa (*Greedy*):** Mantém uma única árvore que cresce a partir de um vértice inicial $r \in V$. Em cada etapa, adiciona a aresta de menor peso que conecta a árvore atual a um novo vértice fora dela.
- **Estrutura de Dados:** Utiliza uma **Fila de Prioridades (*Min-Heap*)** para gerenciar as arestas da fronteira de corte.
- **Complexidade Computacional:** $\mathcal{O}(|E| \log |V|)$ com *Binary Heap* ou $\mathcal{O}(|E| + |V| \log |V|)$ com *Fibonacci Heap*.

In [1]:
# Importação de Bibliotecas Standard e de Terceiros
from typing import List, Tuple, Dict, Set
import heapq
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Configuração estática de visualização
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas carregadas com sucesso.")

---
### 3. Implementação Orientada a Objetos: Estruturas de Dados e Algoritmos

Abaixo definimos a classe `DisjointSet` (Union-Find) e as implementações puras dos algoritmos de **Kruskal** e **Prim** em Python 3.

In [2]:
class DisjointSet:
    """
    Estrutura de Dados Union-Find com Otimizações de:
    - Compressão de Caminho (Path Compression)
    - União por Rank (Union by Rank)
    """
    def __init__(self, vertices: List[str]):
        self.parent: Dict[str, str] = {v: v for v in vertices}
        self.rank: Dict[str, int] = {v: 0 for v in vertices}

    def find(self, item: str) -> str:
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, root1: str, root2: str) -> bool:
        r1 = self.find(root1)
        r2 = self.find(root2)

        if r1 == r2:
            return False

        if self.rank[r1] < self.rank[r2]:
            self.parent[r1] = r2
        elif self.rank[r1] > self.rank[r2]:
            self.parent[r2] = r1
        else:
            self.parent[r2] = r1
            self.rank[r1] += 1
        return True

def kruskal_mst(vertices: List[str], edges: List[Tuple[str, str, float]]) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    sorted_edges = sorted(edges, key=lambda x: x[2])
    dsu = DisjointSet(vertices)

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []

    for u, v, w in sorted_edges:
        root_u = dsu.find(u)
        root_v = dsu.find(v)

        if root_u != root_v:
            dsu.union(u, v)
            mst_edges.append((u, v, w))
            total_weight += w
            execution_log.append({"Passo": len(execution_log) + 1, "Aresta": f"{u} - {v}", "Peso (m)": w, "Status": "INCLUÍDA", "Motivo": "Vértices em componentes separadas"})
        else:
            execution_log.append({"Passo": len(execution_log) + 1, "Aresta": f"{u} - {v}", "Peso (m)": w, "Status": "REJEITADA", "Motivo": "Formaria ciclo"})

        if len(mst_edges) == len(vertices) - 1:
            break
    return mst_edges, total_weight, execution_log

def prim_mst(vertices: List[str], edges: List[Tuple[str, str, float]], start_node: str) -> Tuple[List[Tuple[str, str, float]], float, List[Dict]]:
    adj: Dict[str, List[Tuple[float, str]]] = {v: [] for v in vertices}
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))

    visited: Set[str] = {start_node}
    min_heap: List[Tuple[float, str, str]] = []

    for w, neighbor in adj[start_node]:
        heapq.heappush(min_heap, (w, start_node, neighbor))

    mst_edges: List[Tuple[str, str, float]] = []
    total_weight: float = 0.0
    execution_log: List[Dict] = []
    step = 1

    while min_heap and len(visited) < len(vertices):
        w, u, v = heapq.heappop(min_heap)
        if v in visited:
            execution_log.append({"Iteração": step, "Aresta": f"{u} - {v}", "Peso (m)": w, "Decisão": "IGNORADA", "Motivo": f"{v} já está na árvore"})
            step += 1
            continue

        visited.add(v)
        mst_edges.append((u, v, w))
        total_weight += w
        execution_log.append({"Iteração": step, "Aresta": f"{u} - {v}", "Peso (m)": w, "Decisão": "ADICIONADA", "Motivo": f"Expande árvore para {v}"})
        step += 1

        for next_w, neighbor in adj[v]:
            if neighbor not in visited:
                heapq.heappush(min_heap, (next_w, v, neighbor))
    return mst_edges, total_weight, execution_log

---
### 4. Definição do Grafo da Estação de Hidrogênio (HRS)

O modelo agora utiliza componentes adequados ao sistema da sua HRS, consistindo de 8 unidades operacionais interligadas no sistema SCADA e rotas viáveis de instrumentação:

| Vértice | Nome da Unidade | Setor Operacional |
| :--- | :--- | :--- |
| `ELET-01` | Eletrolisador | Geração |
| `PURIF-01` | Purificador de H2 | Tratamento |
| `TK-BP` | Tanque de Baixa Pressão | Armazenamento |
| `COMP-01` | Compressor Principal | Compressão |
| `TK-AP` | Tanque de Alta Pressão | Armazenamento |
| `RESF-01` | Sistema de Resfriamento (Chiller) | Condicionamento |
| `DISP-01` | Dispenser de Hidrogênio | Abastecimento |
| `SCADA-01` | Centro de Controle SCADA | Automação |

In [3]:
# Lista de Equipamentos (Vértices)
nodes = [
    "ELET-01", "PURIF-01", "TK-BP", "COMP-01",
    "TK-AP", "RESF-01", "DISP-01", "SCADA-01"
]

# Rotas com Distâncias em Metros (Arestas Ponderadas)
edges = [
    ("ELET-01", "PURIF-01", 5.0),
    ("ELET-01", "SCADA-01", 25.0),
    ("PURIF-01", "TK-BP", 10.0),
    ("PURIF-01", "SCADA-01", 20.0),
    ("TK-BP", "COMP-01", 8.0),
    ("TK-BP", "SCADA-01", 15.0),
    ("COMP-01", "TK-AP", 12.0),
    ("COMP-01", "SCADA-01", 18.0),
    ("TK-AP", "RESF-01", 15.0),
    ("TK-AP", "DISP-01", 20.0),
    ("RESF-01", "DISP-01", 5.0),
    ("RESF-01", "SCADA-01", 22.0),
    ("DISP-01", "SCADA-01", 30.0),
    ("TK-BP", "RESF-01", 28.0)
]

print(f"Grafo da HRS carregado: {len(nodes)} Equipamentos e {len(edges)} Conexões viáveis.")

---
### 5. Execução e Validação dos Algoritmos

In [4]:
# Executando Kruskal
kruskal_edges, kruskal_weight, kruskal_log = kruskal_mst(nodes, edges)

# Executando Prim (iniciando no Eletrolisador)
prim_edges, prim_weight, prim_log = prim_mst(nodes, edges, start_node="ELET-01")

print("=" * 60)
print(" RESULTADOS DA OTIMIZAÇÃO TOPOLÓGICA (HRS)")
print("=" * 60)
print(f"Metragem Total de Tubulação/Cabos (Kruskal): {kruskal_weight:.2f} metros")
print(f"Metragem Total de Tubulação/Cabos (Prim):    {prim_weight:.2f} metros")
print(f"Equivalência Estrita de Custo:             {kruskal_weight == prim_weight}")
print("=" * 60)

#### Tabela 1: Histórico de Execução do Algoritmo de Kruskal

In [5]:
df_kruskal = pd.DataFrame(kruskal_log)
display(df_kruskal)

#### Tabela 2: Histórico de Expansão do Algoritmo de Prim

In [6]:
df_prim = pd.DataFrame(prim_log)
display(df_prim)

---
### 6. Visualização Gráfica do Rede Industrial e Topologia MST

In [7]:
def plot_mst(nodes, edges, mst_edges, title):
    G = nx.Graph()
    G.add_nodes_from(nodes)
    for u, v, w in edges:
        G.add_edge(u, v, weight=w)

    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(10, 6))

    # Arestas normais (tracejadas) e da MST (sólidas)
    nx.draw_networkx_edges(G, pos, edgelist=edges, style='dashed', alpha=0.3, edge_color='gray')
    nx.draw_networkx_edges(G, pos, edgelist=mst_edges, width=2.5, edge_color='green')

    # Nós e identificações
    nx.draw_networkx_nodes(G, pos, node_size=1500, node_color='lightblue', edgecolors='black')
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

    # Pesos das arestas
    edge_labels = {(u, v): f"{w}m" for u, v, w in edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.show()

plot_mst(nodes, edges, kruskal_edges, "Topologia Otimizada da HRS (Árvore Geradora Mínima)")